# 09 · Delta Lake — ACID & Time Travel Demo

## Section 1 — Setup

In [0]:
# Setup: point at your Silver table for the demo
CATALOG = "vstone_catalog"
SILVER  = "silver"
FQN     = f"{CATALOG}.{SILVER}.listings_silver_merged"   # fqn - fully qualified name

# Find the latest clean DLT version (used later for restore demo)
history = spark.sql(f"DESCRIBE HISTORY {FQN}").select("version","operation") \
    .orderBy("version", ascending=False).collect()

CLEAN_VERSION = next((r["version"] for r in history if r["operation"] == "STREAMING UPDATE"), None)

print(f"Table          : {FQN}")
print(f"Clean version  : {CLEAN_VERSION}  ← last STREAMING UPDATE (DLT baseline)")
print()
print("Use this version number in the Time Travel cells below.")
print(f"  SQL: SELECT * FROM {FQN} VERSION AS OF {CLEAN_VERSION}")

## Section 2 — ACID: Atomicity

In [0]:
%sql
-- Create a small demo Delta table for our ACID experiments
-- (keeps the demo isolated from production Silver data)
CREATE OR REPLACE TABLE vstone_catalog.silver.acid_demo
(
  id       BIGINT,
  brand    STRING,
  price    DOUBLE,
  status   STRING
)
USING DELTA
COMMENT 'Temporary table for ACID + Time Travel demo';

-- Confirm it exists
SHOW TABLES IN vstone_catalog.silver LIKE 'acid_demo';

In [0]:
%sql
-- ATOMICITY DEMO: INSERT 5 rows in one atomic transaction
-- Either ALL 5 land, or NONE do. Delta never leaves partial results.
INSERT INTO vstone_catalog.silver.acid_demo VALUES
  (1, 'toyota',    850000,  'active'),
  (2, 'honda',     620000,  'active'),
  (3, 'bmw',      1800000,  'active'),
  (4, 'lada',      180000,  'active'),
  (5, 'mercedes', 2500000,  'active');

-- Verify: all 5 landed atomically
SELECT 'Rows after atomic INSERT' AS event, COUNT(*) AS row_count
FROM vstone_catalog.silver.acid_demo;

## Section 3 — ACID: Consistency

In [0]:
%sql
-- CONSISTENCY DEMO: UPDATE only valid rows
-- Delta enforces the predicate atomically — only matching rows change
UPDATE vstone_catalog.silver.acid_demo
SET    status = 'premium'
WHERE  price  > 1000000;

-- Result: toyota/bmw/mercedes flagged as premium, others unchanged
SELECT id, brand, price, status
FROM   vstone_catalog.silver.acid_demo
ORDER  BY id;

In [0]:
%sql
-- CONSISTENCY DEMO: DELETE with a precise predicate
-- Only the exact matching rows are removed — nothing else is touched
DELETE FROM vstone_catalog.silver.acid_demo
WHERE brand = 'lada';

-- Confirm: 4 rows remain, lada is gone
SELECT 'Rows after DELETE of lada' AS event, COUNT(*) AS row_count
FROM   vstone_catalog.silver.acid_demo;

## Section 4 — ACID: Isolation


In [0]:
%sql
-- ISOLATION DEMO: Snapshot isolation in action
-- This SELECT always returns the committed snapshot at query start time.
-- Even if another session is writing concurrently, you see a clean snapshot.

-- Reader sees Version N (committed snapshot)
SELECT 'Snapshot isolation — reader sees committed state' AS isolation_proof,
       COUNT(*)   AS row_count,
       MIN(price) AS min_price,
       MAX(price) AS max_price
FROM   vstone_catalog.silver.acid_demo;

In [0]:
%sql
-- ISOLATION DEMO: Read a specific older version explicitly
-- This is time-travel used for isolation testing:
-- "What did the table look like before the DELETE?"
SELECT 'Version 1 snapshot (before DELETE)' AS snapshot, id, brand, price, status
FROM   vstone_catalog.silver.acid_demo VERSION AS OF 1
ORDER  BY id;

## Section 5 — ACID: Durability

In [0]:
%sql
-- DURABILITY PROOF: The transaction log records every commit permanently.
-- Version number, timestamp, operation, and file changes — all durable.
DESCRIBE HISTORY vstone_catalog.silver.acid_demo;

In [0]:
%sql
-- DURABILITY: Count how many versions (commits) have been durably recorded
SELECT COUNT(*) AS total_durable_commits
FROM (DESCRIBE HISTORY vstone_catalog.silver.acid_demo);

## Section 6 — Time Travel: Browse History

In [0]:
%sql
-- TIME TRAVEL SETUP: See all available versions of our Silver table
-- version = the version number you can use in VERSION AS OF
-- operation = what created this version (STREAMING UPDATE, MERGE, UPDATE, etc.)
DESCRIBE HISTORY vstone_catalog.silver.listings_silver_merged;

In [0]:
%sql
-- TIME TRAVEL: How many versions exist? What's the current version?
SELECT
  COUNT(*)   AS total_versions,
  MAX(version) AS current_version,
  MIN(version) AS oldest_version,
  MIN(timestamp) AS oldest_timestamp
FROM (DESCRIBE HISTORY vstone_catalog.silver.listings_silver_merged);

## Section 10 — Full Audit Trail on Silver

In [0]:
%sql
-- AUDIT TRAIL: Full history of the production Silver table
-- operation = what type of change
-- userName  = who made it (IAM identity)
-- operationMetrics = rows written, files added/removed
SELECT
  version,
  timestamp,
  userName,
  operation,
  operationParameters,
  operationMetrics
FROM (DESCRIBE HISTORY vstone_catalog.silver.listings_silver_merged)
ORDER BY version DESC;

In [0]:
%sql
-- AUDIT TRAIL: Summary — which operations created versions and when?
SELECT
  operation,
  COUNT(*)         AS num_commits,
  MIN(timestamp)   AS first_seen,
  MAX(timestamp)   AS last_seen
FROM (DESCRIBE HISTORY vstone_catalog.silver.listings_silver_merged)
GROUP BY operation
ORDER BY num_commits DESC;

## Section 11 — Business Rule Validation on Silver

In [0]:
%sql
-- RULE 1: listing_year and listing_month must match listing_date
-- Zero violations expected
SELECT
  'listing_year/month match listing_date' AS rule,
  COUNT(*) AS violations
FROM vstone_catalog.silver.listings_silver_merged
WHERE listing_date IS NOT NULL
  AND (
    YEAR(listing_date)  != listing_year
    OR MONTH(listing_date) != listing_month
  );

In [0]:
%sql
-- RULE 2: price_usd = ROUND(price_rub / 82.5, 2)
-- USD rate fixed at Feb 2023 historical rate
SELECT
  'price_usd = ROUND(price_rub / 82.5, 2)' AS rule,
  COUNT(*) AS violations
FROM vstone_catalog.silver.listings_silver_merged
WHERE price_rub IS NOT NULL
  AND price_usd IS NOT NULL
  AND ABS(price_usd - ROUND(price_rub / 82.5, 2)) > 0.01;

In [0]:
%sql
-- RULE 3: price_category must match the correct RUB band
-- BUDGET < 300K | MID_RANGE 300K-700K | PREMIUM 700K-1.5M | LUXURY > 1.5M
SELECT
  price_category,
  COUNT(*)         AS row_count,
  MIN(price_rub)   AS min_rub,
  MAX(price_rub)   AS max_rub
FROM vstone_catalog.silver.listings_silver_merged
WHERE price_rub IS NOT NULL
GROUP BY price_category
ORDER BY min_rub;

In [0]:
%sql
-- RULE 4: car_age_years = 2023 - manufacture_year
-- Spot-check derivation against manufacture_year
SELECT
  'car_age_years = 2023 - manufacture_year' AS rule,
  COUNT(*) AS violations
FROM vstone_catalog.silver.listings_silver_merged
WHERE manufacture_year IS NOT NULL
  AND car_age_years != (2023 - manufacture_year);

In [0]:
%sql
-- RULE 5: No NULL listing_id or price_rub in Silver (quarantine should catch them)
SELECT
  'NULL listing_id in Silver' AS check_name,
  COUNT(*) AS count
FROM vstone_catalog.silver.listings_silver_merged
WHERE listing_id IS NULL
UNION ALL
SELECT
  'NULL price_rub in Silver',
  COUNT(*)
FROM vstone_catalog.silver.listings_silver_merged
WHERE price_rub IS NULL
UNION ALL
SELECT
  'NULL listing_date in Silver',
  COUNT(*)
FROM vstone_catalog.silver.listings_silver_merged
WHERE listing_date IS NULL;